Name :


E-mail :

# Useful Jupyter Notebook Shortcuts

Here are some helpful keyboard shortcuts for Jupyter Notebook:

- **M**: Switch to Markdown mode
- **Y**: Switch to Code mode
- **A**: Insert cell above
- **B**: Insert cell below
- **D, D**: (Press D twice) Delete selected cell
- **Shift + Enter**: Run the current cell and move to the next
- **Ctrl + Enter**: Run the current cell and stay on it
- **Shift + Tab**: Show function/method documentation
- **Ctrl + Shift + -**: Split cell at cursor
- **Esc**: Enter command mode (blue border)
- **Enter**: Enter edit mode (green border)


## default imports

In [2]:
# import for internal use
from urllib.error import URLError
import os
from pathlib import Path
from keras.utils import get_file

In [3]:
# general imports
import pandas as pd
import numpy as np
from numpy.fft import rfft, irfft, rfftfreq
import matplotlib.pyplot as plt

In [4]:
# machine learning imports
import tensorflow as tf
from keras.layers import Input, Dense,  Concatenate, Activation, Add, Flatten, Reshape
from keras.models import Model
from keras import optimizers, regularizers



## Public Functions

In [5]:
def load_weather_dataset(
    url: str,
    fname: str
    ) -> pd.DataFrame:
    """
    Load the weather data and return a pandas dataframe.

    This function downloads a dataset, extracts it, and creates a pandas dataframe.

    Parameters
    ----------
    url : str
        URL or local path to the csv file.
    fname : str
        Filename to save the downloaded dataset.

    Returns
    -------
    DataFrame : A Pandas DataFrame containing aggregated data

    Examples
    --------
    >>> url = "https://example.com/dataset.zip"
    >>> fname = "weather_data.csv"
    >>> weather_df = load_and_create_datasets(url, fname)
    >>> print(weather_df.head(5))

    Raises
    ------
    Exception
        If the URL download fails, it attempts to use the URL as a local filename.

    See Also
    --------
    keras.utils.get_file : Used for downloading and extracting the dataset
    """


    FileHash = r"dc860688aeb26bee5c4b7d60397274ce499aaf95211c57ab9319cbc86757f592"

    try:
        dset_download = get_file(fname=fname,
            origin=url,
            file_hash=FileHash,
            force_download=False
        )
    except Exception:
            print("URL download failed, try using DATASET_URL as a local filename")
            dset_download = get_file(fname=fname,
                                          origin  = "file:\\"+url,
            file_hash=FileHash,
            force_download=False
        )


    # Step 2: Get the path of the extracted directory
    dataset_dir = os.path.dirname(dset_download)
    weather_df = pd.read_csv(os.path.join(dataset_dir, fname))
    weather_df.rename(columns={"Unnamed: 0":"week_number"}, inplace=True)

    print("Data downloaded and pandas dataframe created")
    return weather_df

In [6]:
def generate_time_signal()-> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Generate a time signal and its forecast.

    This function creates a composite signal consisting of two sine waves and a linear trend.
    It generates the signal for two time ranges: the initial signal and a forecast.

    Returns
    -------
    tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]
        A tuple containing four NumPy arrays:
        - t : np.ndarray
            Time points for the initial signal (0 to 1 seconds, 128 points).
        - y : np.ndarray
            Initial signal values corresponding to 't'.
        - t_forecast : np.ndarray
            Time points for the forecast (1 to 3 seconds, 256 points).
        - y_forecast : np.ndarray
            Forecast signal values corresponding to 't_forecast'.


    The initial signal is sampled at 128 points over 0 to 1 second.
    The forecast is sampled at 256 points over 1 to 3 seconds.
    """
    def signal(t):
        return np.sin(4.25*np.pi*t) + np.sin(8.5*np.pi*t) + 5*t
    t = np.linspace(0,1, num=128)
    t_forecast = np.linspace(1,3, num=256)
    y = signal(t)
    y_forecast = signal(t_forecast)
    return t, y, t_forecast, y_forecast


In [25]:
def fft_forecast(signal:np.ndarray, upsamplig:int=3, exclude_signal:bool=True)->np.ndarray:
  """
  Perform FFT-based forecasting on a given signal.

  This function applies Fast Fourier Transform (FFT) to the input signal,
  performs zero-padding in the frequency domain for upsampling,
  and then applies Inverse Fast Fourier Transform (IFFT) to obtain
  the upsampled signal.

  Parameters
  ----------
  signal : np.ndarray
      The input signal for forecasting.
  upsamplig : int, optional
      The upsampling factor. The output signal will have
      `upsamplig` times more points than the input signal.
      Defaults to 3.
  exclude_signal : bool, optional
      If True, the original signal portion is excluded from the output,
      returning only the forecasted part. Defaults to True.

  Returns
  -------
  np.ndarray
      The forecasted signal.

  Examples
  --------
  >>> import numpy as np
  >>> from numpy.fft import rfft, irfft
  >>> signal = np.sin(np.linspace(0, 2*np.pi, 10))
  >>> forecast = fft_forecast(signal, upsamplig=2, exclude_signal=True)
  >>> print(forecast.shape)
  (10,)

  """
  fft_y = rfft(signal)


  # Create an extended frequency domain array with zeros
  # The length is (len(fft_y) - 1) * n + 1 to accommodate the upsampled signal
  # the dtype argument has to be put otherwise only the real part of the fft will be stored after
  ext_f = np.zeros(((len(fft_y) - 1) * upsamplig + 1,), dtype=fft_y.dtype)

  # Copy the original frequency components to the extended array
  # Scale the amplitude by n to preserve the signal energy
  ext_f[::upsamplig] = fft_y * upsamplig
  ext_y = irfft(ext_f)

  # if True, remove the original signal from the computed one
  if exclude_signal:
    ext_y = ext_y[len(signal):]

  return ext_y

## Global Variables

In [9]:
WEATHER_DATASET_URL = r"https://amubox.univ-amu.fr/s/tCEGCpYMk5ANCRT/download/weather_jena_2004_2023.csv"
WEATHER_DATASET_FNAME = "weather_jena_2004-2023.csv"

# Test Signal

### Signal Generation

In [13]:
t, y, t_forecast, y_forecast = generate_time_signal()

## Fourier Analysis

## Fourier Forecasting

## Fourier Network Implementation and Training

## Fourier Network Prediction

## Initial Weights Implementation

## Model Interpretability

# Weather data Loading

In [10]:
weather_df = load_weather_dataset(WEATHER_DATASET_URL, WEATHER_DATASET_FNAME)

125421/125421 ━━━━━━━━━━━━━━━━━━━━ 0s 3us/step
Data downloaded and pandas dataframe created


,week_number,date,T_mean,T_min,T_max,p_mean,p_min,p_max,rho_mean,rho_min,rho_max,year,week
0,0,2004-01-04,-4.336052,-9.71,0.04,991.806904,985.26,997.93,1283.474678,1253.53,1312.08,2004,1
1,1,2004-01-11,1.387659,-8.19,9.28,986.651409,974.00,997.97,1249.345427,1198.20,1309.98,2004,2
2,2,2004-01-18,3.421766,-3.57,8.67,973.310278,958.21,997.53,1223.163671,1181.79,1286.46,2004,3
3,3,2004-01-25,-2.463452,-8.07,3.90,988.472966,972.13,996.67,1270.189861,1232.71,1304.77,2004,4
4,4,2004-02-01,1.083105,-5.77,11.76,978.643919,966.88,993.15,1241.069683,1197.71,1268.53,2004,5


## Fourier Network Train and Prédict

## Comparison with Fourier forecasting

## Conclustions

### References